# Week 55, Day 3: The lasso: sparsity by soft thresholding

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w055/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Three ways to shrink

Write soft_threshold(z, lam). Check that, for a design with orthonormal columns scaled so that XᵀX = nI, soft-thresholding the least-squares estimates gives a point that no small step in any coordinate can improve. Then plot the soft, hard and ridge rules.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(553)

def soft_threshold(z, lam):
    return ...

nq = 50
Q = np.linalg.qr(r.normal(size=(nq, 5)))[0] * np.sqrt(nq)     # orthogonal columns with Q'Q = n I
yq = Q @ np.array([1.0, 0.5, 0.1, 0.0, -0.3]) + 0.3 * r.normal(size=nq)
z = Q.T @ yq / nq
print("least-squares estimates:", np.round(z, 3))
print("lasso (lambda = 0.2):  ", np.round(soft_threshold(z, 0.2), 3))
zz = np.linspace(-3, 3, 201)
plt.plot(zz, soft_threshold(zz, 1), label="soft (lasso)"); plt.plot(zz, np.where(np.abs(zz) > 1, zz, 0), label="hard (best subset)")
plt.plot(zz, zz / 2, label="ridge"); plt.plot(zz, zz, "k:", lw=0.8); plt.xlabel("least-squares estimate"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_soft():
    "soft_threshold implements sign(z) max(|z| - lam, 0)"
    z = np.array([-3.0, -0.5, 0.0, 0.4, 2.0])
    assert np.allclose(soft_threshold(z, 1.0), [-2.0, 0.0, 0.0, 0.0, 1.0])
def test_orth():
    "for an orthonormal design, the soft-thresholded least-squares estimates solve the lasso"
    w = soft_threshold(Q.T @ yq / nq, 0.2)
    obj = lambda v: 0.5 / nq * np.sum((yq - Q @ v) ** 2) + 0.2 * np.abs(v).sum()
    assert all(obj(w) <= obj(w + 1e-3 * e) + 1e-12 for e in np.r_[np.eye(5), -np.eye(5)])

test_soft()
test_orth()
print("All checks passed")

### Exercise: The lasso by coordinate descent

Complete lasso_cd for standardised columns (each with mean 0 and mean square 1): for each j, compute rho = x_j · (partial residual) / n and set w_j = soft_threshold(rho, lam), updating the residual as you go. Then check the optimality (KKT) conditions: for non-zero weights the gradient of the squared-error term equals lam times the sign, and for zero weights its size is at most lam.

In [ ]:
n, P = 100, 50
X = r.normal(size=(n, P)); X[:, 1] = 0.7 * X[:, 0] + 0.7 * X[:, 1]
X = (X - X.mean(0)) / X.std(0)
w_true = np.zeros(P); w_true[[0, 3, 7, 12, 20]] = [1.0, -0.8, 0.6, 0.5, -0.4]
y = X @ w_true + 0.5 * r.normal(size=n); y = y - y.mean()

def lasso_cd(X, y, lam, iters=200, w0=None):
    n = len(y)
    w = np.zeros(X.shape[1]) if w0 is None else w0.copy()
    res = y - X @ w
    for it in range(iters):
        for j in range(X.shape[1]):
            rho = X[:, j] @ (res + X[:, j] * w[j]) / n
            new = ...
            res -= X[:, j] * (new - w[j])
            w[j] = new
    return w

w = lasso_cd(X, y, 0.1)
grad = X.T @ (y - X @ w) / n
print("non-zero weights at indices:", np.flatnonzero(w), "values", np.round(w[w != 0], 2))
print(f"largest |gradient| among zero weights: {np.abs(grad[w == 0]).max():.4f} (must not exceed 0.1)")

In [ ]:
import numpy as np
def test_objective():
    "lasso_cd reaches a lower objective than the starting point and its result satisfies the optimality conditions"
    w = lasso_cd(X, y, 0.1)
    g = X.T @ (y - X @ w) / len(y)
    nz = w != 0
    assert np.allclose(g[nz], 0.1 * np.sign(w[nz]), atol=1e-4) and np.all(np.abs(g[~nz]) <= 0.1 + 1e-4)
def test_sparse():
    "with lambda = 0.1 most weights are exactly zero"
    assert np.sum(lasso_cd(X, y, 0.1) == 0) >= 30

test_objective()
test_sparse()
print("All checks passed")

### Exercise: Selection accuracy across penalties

For each penalty, fit the lasso to 15 fresh datasets (same design, new noise and new random inputs), and record the mean number of the 5 true inputs selected (tp) and of the 45 others selected (fp).

In [ ]:
lams = [0.02, 0.05, 0.1, 0.2, 0.35]
truth = w_true != 0
tp, fp = np.zeros(len(lams)), np.zeros(len(lams))
for k, lam in enumerate(lams):
    t_, f_ = [], []
    for rep in range(15):
        Xr = r.normal(size=(n, P)); Xr = (Xr - Xr.mean(0)) / Xr.std(0)
        yr = Xr @ w_true + 0.5 * r.normal(size=n); yr = yr - yr.mean()
        sel = lasso_cd(Xr, yr, lam, iters=50) != 0
        t_.append(...)
        f_.append(...)
    tp[k], fp[k] = np.mean(t_), np.mean(f_)
    print(f"lambda {lam:.2f}: true inputs found {tp[k]:.2f} of 5, false inputs {fp[k]:.2f} of 45")

In [ ]:
import numpy as np
def test_tp():
    "at the smallest penalty all five true inputs are almost always found"
    assert tp[0] > 4.8
def test_fp():
    "false positives fall as the penalty rises, and true positives eventually fall too"
    assert fp[0] > fp[-1] and tp[-1] < tp[0]

test_tp()
test_fp()
print("All checks passed")